# 05. 날짜·시간 다루기 · `.dt` 접근자 🕒

어제 문자열을 `.str` 로 정리했죠. 오늘은 그 짝꿍인 **`.dt` 접근자**입니다.

`'2025-03-13 08:15'` 은 우리 눈엔 시간이지만, 판다스에겐 아직 **그냥 글자**예요.
`pd.to_datetime()` 으로 **진짜 날짜형**으로 바꾸면, 그때부터 `.dt` 로 연·월·일·시·요일을
**한 번에** 뽑아낼 수 있습니다.

> 🔑 `.str` 은 글자 컬럼, `.dt` 는 날짜 컬럼에 붙는 **쌍둥이 접근자**입니다.
> 둘 다 `for` 문 없이 컬럼 전체를 한 방에 처리해요.

같은 **카페 주문 로그**로, 이번엔 `주문일시` 를 파헤쳐 봅니다.

## 0단계 · 데이터 불러오기 (제공)

In [ ]:
# [제공] 라이브러리 & 데이터 로드
import pandas as pd

df = pd.read_csv('data/주문로그.csv', encoding='utf-8-sig')
# 지금 '주문일시'는 날짜가 아니라 '문자열(object)' 입니다 ↓ 확인!
print(df['주문일시'].dtype)
df[['주문번호', '주문일시']].head()

## 🎯 미션 1 · 문자열 → 진짜 날짜로 `pd.to_datetime()`

`주문일시` 컬럼을 `pd.to_datetime()` 으로 변환해 다시 저장하고,
자료형이 `object` → `datetime64` 로 바뀌었는지 확인하세요.

In [ ]:
# [미션 1] 날짜형으로 변환
#  df['주문일시'] = pd.to_datetime(df['주문일시'])
df['주문일시'] = pd.to_datetime(df['주문일시'])
print('변환 후 dtype:', df['주문일시'].dtype)   # datetime64[ns]
df['주문일시'].head()

## 🎯 미션 2 · 연·월·일·시 뽑아내기 `.dt.year / month / hour`

날짜형이 되면 `.dt` 로 성분을 꺼낼 수 있습니다.
`월`(`.dt.month`)과 `시`(`.dt.hour`) 컬럼을 만드세요.

In [ ]:
# [미션 2] 월, 시(시간) 파생
#  df['월'] = df['주문일시'].dt.month
#  df['시'] = df['주문일시'].dt.hour
df['월'] = df['주문일시'].dt.month
df['시'] = df['주문일시'].dt.hour
df[['주문일시', '월', '시']].head()

## 🎯 미션 3 · 주말 주문일까? `.dt.dayofweek`

`.dt.dayofweek` 는 요일을 숫자로 줍니다 (월=0 … 토=5, 일=6).
**토·일이면 True** 인 `주말여부` 컬럼을 만드세요. (힌트: `>= 5`)

In [ ]:
# [미션 3] 주말 여부 (토=5, 일=6)
#  df['주말여부'] = df['주문일시'].dt.dayofweek >= 5
df['주말여부'] = df['주문일시'].dt.dayofweek >= 5
df[['주문일시', '주말여부']].head()

## 🎯 미션 4 · '연-월' 라벨 만들기 `.dt.strftime()`

월별 집계에 쓰기 좋은 `'2025-03'` 형태의 `연월` 컬럼을 만드세요.
`.dt.strftime('%Y-%m')` 을 사용합니다.

In [ ]:
# [미션 4] '2025-03' 형태의 연월 라벨
#  df['연월'] = df['주문일시'].dt.strftime('%Y-%m')
df['연월'] = df['주문일시'].dt.strftime('%Y-%m')
df['연월'].value_counts()

## 🧩 파생 도우미 · 한글 요일 & 시간대 (제공)
요일을 한글로, 시간을 오전/오후/저녁으로 묶어 둡니다. 내일 `groupby` 에서 바로 씁니다.

In [ ]:
# [제공] 한글 요일 + 시간대 구간
#  .map(대응표) : 숫자 요일 → 한글 요일
요일한글 = {0: '월', 1: '화', 2: '수', 3: '목', 4: '금', 5: '토', 6: '일'}
df['요일'] = df['주문일시'].dt.dayofweek.map(요일한글)

#  pd.cut : 숫자(시)를 구간으로 나눠 라벨 붙이기
df['시간대'] = pd.cut(df['시'],
                    bins=[0, 12, 17, 24],
                    labels=['오전', '오후', '저녁'],
                    right=False)

df[['주문일시', '요일', '시', '시간대']].head()

## ✅ 파생하니 이런 게 보인다 (미리보기)
날짜 하나에서 뽑아낸 컬럼들로 벌써 패턴이 보입니다. (본격 요약은 내일 `groupby`에서!)

In [ ]:
# 요일별 / 시간대별 주문 건수
print('▼ 요일별 주문 수')
print(df['요일'].value_counts())
print()
print('▼ 시간대별 주문 수')
print(df['시간대'].value_counts())

---
### 💡 오늘 배운 `.dt` 한눈에

| 코드 | 하는 일 | 결과 예 |
|------|---------|---------|
| `pd.to_datetime(s)` | 문자열 → 날짜형 | `'2025-03-13'` → `Timestamp` |
| `.dt.year / month / day` | 연·월·일 | `2025`, `3`, `13` |
| `.dt.hour` | 시(시간) | `8` |
| `.dt.dayofweek` | 요일 숫자(월0~일6) | `3` (목) |
| `.dt.strftime('%Y-%m')` | 원하는 형식의 문자열 | `'2025-03'` |

> ✨ 핵심 : **먼저 `to_datetime` 으로 날짜형으로 바꿔야** `.dt` 를 쓸 수 있습니다.
> 문자열(`.str`) → 날짜(`.dt`) 로 이어지는 흐름을 기억하세요.
